In [84]:
import pandas as pd
from pathlib import Path
import numpy as np

In [85]:
REVIEWS = Path("../reseñas")
en_reviews = REVIEWS / "resenas_ingles.csv"
es_reviews = REVIEWS / "resenas_espanol_trad.csv"
pt_reviews = REVIEWS / "resenas_portugues_trad.csv"

In [86]:
N = 2100
one_third = int(N/3)

In [87]:
en_df = pd.read_csv(en_reviews)
es_df = pd.read_csv(es_reviews)
pt_df = pd.read_csv(pt_reviews)

In [88]:
en_df.head(20)

,Rating,Date,Review Text
0,5,4/8/2016,Tonight was my first time coming here and it w...
1,5,4/13/2019,All I can say about this place is... you must ...
2,4,4/11/2016,This place is near Fremont street. Had to pay ...
3,2,6/2/2016,While I am glad I can say I've tried Sprinkles...
4,4,2/28/2019,I lived in New York City nearly 20 years and t...
5,1,3/1/2016,This place was HORRIBLE! We got the crumb cake...
6,4,8/22/2020,I've been a fan of Hopscotch since they opened...
7,3,1/6/2015,Waited over an hour in line to get some Cupcak...
8,5,3/21/2022,Delicious. We got a breakfast sandwich here an...
9,4,1/23/2021,This milkshake was prettier than it tasted. I ...


In [89]:
en_df.head()

,Rating,Date,Review Text
0,5,4/8/2016,Tonight was my first time coming here and it w...
1,5,4/13/2019,All I can say about this place is... you must ...
2,4,4/11/2016,This place is near Fremont street. Had to pay ...
3,2,6/2/2016,While I am glad I can say I've tried Sprinkles...
4,4,2/28/2019,I lived in New York City nearly 20 years and t...


In [90]:
SEED = 42
rng = np.random.default_rng(SEED)

used = set()
def sample(df, text_col):
    pool = df[~df["Review Text"].isin(used)].drop_duplicates(subset=[text_col])
    picked = pool.iloc[rng.choice(len(pool), one_third, replace=False)]
    used.update(picked["Review Text"])
    return picked

selected_en = sample(en_df, "Review Text").drop(columns=["Date", "Rating"])
selected_es = sample(es_df, "Review Text_es").drop(columns=["Review Text", "Date", "Rating"])
selected_pt = sample(pt_df, "Review Text_pt-BR").drop(columns=["Review Text", "Date", "Rating"])

In [91]:
selected_en["lang"] = "en"
selected_es["lang"] = "es"
selected_pt["lang"] = "pt"

In [92]:
selected_es = selected_es.rename(columns={"Review Text_es": "text"})
selected_pt = selected_pt.rename(columns={"Review Text_pt-BR": "text"})
selected_en = selected_en.rename(columns={"Review Text": "text"})

In [93]:
df = pd.concat([selected_es, selected_en, selected_pt])
df

,text,lang
4850,"Todo lo que tienes que decir es ""Helado de Bab...",es
124,Este lugar manejó muy bien la situación durant...,es
5624,Pasadas unas semanas desde que regresé de mis ...,es
626,Vine aquí porque realmente quería probar sus d...,es
5496,"Quiero decirles, fue un verdadero placer descu...",es
...,...,...
3012,Já não é tão impressionante. A Tosi se liberto...,pt
4649,Jarlings ocupa um lugar especial em nossos cor...,pt
2616,Eu e meus filhos fomos para o almoço. Embora e...,pt
6246,Os biscoitos são gostosos e têm um sabor parec...,pt


In [94]:
assert not df.duplicated(subset=["text", "lang"]).any(), "duplicate reviews in subset"
assert (df["lang"].value_counts() == one_third).all()
df["lang"].value_counts()

lang
es    700
en    700
pt    700
Name: count, dtype: int64

In [95]:
df[df["lang"] == "pt"].head()

,text,lang
1142,Incrível! Fui depois de conhecer alguns dos ba...,pt
757,Estou adorando todas as novas empresas que est...,pt
6334,Meu amigo recomendou fortemente a La Pasticcer...,pt
2993,O milkshake de canela estava incrível!!! E tam...,pt
5211,A fila tinha cerca de 25 pessoas. O atendiment...,pt


In [96]:
df.to_csv(REVIEWS / "training_subset.csv", index=False)